# 03b — Scenario B: QUBO Construction + Classical Baseline

This notebook takes the Scenario B candidate pool and:

1) Loads `data/processed/scenario_B_candidates.csv`
2) Constructs a QUBO for selecting a portfolio of trials
3) Solves a classical baseline (greedy + optional exact if small)
4) Writes artifacts used by downstream QAOA/Braket notebooks

Outputs:
- `data/qubo_scenarios/scenario_B_qubo.json`
- `data/results/scenario_B_classical_summary.csv`
- `data/results/scenario_B_classical_selected_trials.csv`
- `data/results/scenario_B_classical_best_bitstring.txt`


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and output directories
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# --- Directories ---
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
QUBO_DIR = DATA_DIR / "qubo_scenarios"
RESULTS_DIR = DATA_DIR / "results"

for d in [PROCESSED_DIR, QUBO_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Inputs ---
PATH_CANDIDATES = PROCESSED_DIR / "scenario_B_candidates.csv"

# --- Outputs ---
PATH_QUBO_JSON = QUBO_DIR / "scenario_B_qubo.json"

PATH_CLASSICAL_SUMMARY = RESULTS_DIR / "scenario_B_classical_summary.csv"
PATH_CLASSICAL_SELECTED = RESULTS_DIR / "scenario_B_classical_selected_trials.csv"
PATH_CLASSICAL_BITSTRING = RESULTS_DIR / "scenario_B_classical_best_bitstring.txt"

# --- Require inputs ---
if not PATH_CANDIDATES.exists():
    raise FileNotFoundError(f"Missing required file: {PATH_CANDIDATES}\nRun 06a first.")

print("OK: Found Scenario B candidates:", PATH_CANDIDATES)
print("Outputs will write to:")
print("  -", PATH_QUBO_JSON)
print("  -", PATH_CLASSICAL_SUMMARY)
print("  -", PATH_CLASSICAL_SELECTED)
print("  -", PATH_CLASSICAL_BITSTRING)


OK: Found Scenario B candidates: data/processed/scenario_B_candidates.csv
Outputs will write to:
  - data/qubo_scenarios/scenario_B_qubo.json
  - data/results/scenario_B_classical_summary.csv
  - data/results/scenario_B_classical_selected_trials.csv
  - data/results/scenario_B_classical_best_bitstring.txt


### What Cell 1 Just Did

- Set up canonical input/output paths for Scenario B QUBO construction and baseline solving.
- Confirmed the Scenario B candidates artifact exists before continuing.


In [2]:
# ============================================================
# Cell 2 — Load candidates + enforce required scoring columns
# ============================================================

candidates = pd.read_csv(PATH_CANDIDATES)

required = ["nct_id", "_benefit_raw", "_cost_raw", "_safety_raw"]
missing = [c for c in required if c not in candidates.columns]
if missing:
    raise ValueError(
        "Scenario B candidates missing required columns:\n"
        f"Missing: {missing}\n"
        f"Found: {list(candidates.columns)}\n"
        "Fix: ensure 06a writes _benefit_raw/_cost_raw/_safety_raw."
    )

# Clean / coerce numeric fields
for col in ["_benefit_raw", "_cost_raw", "_safety_raw"]:
    candidates[col] = pd.to_numeric(candidates[col], errors="coerce").fillna(0.0).astype(float)

candidates["nct_id"] = candidates["nct_id"].astype(str)
candidates = candidates.drop_duplicates(subset=["nct_id"]).reset_index(drop=True)

print("Loaded candidates:", candidates.shape)
display(candidates.head(8))


Loaded candidates: (60, 9)


,nct_id,brief_title,lead_sponsor,overall_status,phase,_benefit_raw,_cost_raw,_safety_raw,_pool_score
0,NCT06760637,Study of PF-07220060 With Letrozole in Adults ...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
1,NCT06713616,PCORI Comparative Effectiveness Study-Esketami...,Yale University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
2,NCT06711887,Phase III Extension Study of Efficacy and Safe...,Novartis Pharmaceuticals,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
3,NCT06706817,A Study to Investigate Changes in Symptoms in ...,AstraZeneca,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
4,NCT05674305,Radiotherapy Alone Versus Concurrent Chemo-rad...,Fudan University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
5,NCT06703476,A Study of Surgical Techniques During Cystectomy,Memorial Sloan Kettering Cancer Center,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
6,NCT05675410,A Study to Compare Standard Therapy to Treat H...,National Cancer Institute (NCI),RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
7,NCT06701331,Safety and Efficacy of Upadacitinib in Combina...,AbbVie,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893


### What Cell 2 Just Did

- Loaded `scenario_B_candidates.csv`.
- Enforced a strict schema required for optimization (ID + benefit/cost/safety components).
- Cleaned numeric columns and removed duplicate trial IDs to keep the QUBO deterministic.


In [3]:
# ============================================================
# Cell 3 — Define the optimization objective + build QUBO matrix Q
# ============================================================

# ---- Optimization knobs ----
K = 10                 # portfolio size target (can sweep later)
LAMBDA_COST = 1.0
LAMBDA_SAFETY = 1.0
PENALTY_A = 10.0       # constraint penalty strength (must dominate objective scale)

# ---- Build linear weights for minimization ----
# We minimize:  -benefit + lambda_cost*cost + lambda_safety*safety  (+ penalties)
benefit = candidates["_benefit_raw"].to_numpy(dtype=float)
cost    = candidates["_cost_raw"].to_numpy(dtype=float)
safety  = candidates["_safety_raw"].to_numpy(dtype=float)

w = (-benefit) + (LAMBDA_COST * cost) + (LAMBDA_SAFETY * safety)

n = len(candidates)
Q = np.zeros((n, n), dtype=float)

# Base objective: sum_i w_i x_i -> add to diagonal
for i in range(n):
    Q[i, i] += w[i]

# Cardinality penalty: A * (sum x_i - K)^2
# expands to: A*(sum x_i)^2 - 2AK*(sum x_i) + A*K^2
# with (sum x_i)^2 = sum x_i + 2*sum_{i<j} x_i x_j
for i in range(n):
    Q[i, i] += PENALTY_A * (1 - 2*K)   # from A*sum x_i  and -2AK*sum x_i
for i in range(n):
    for j in range(i+1, n):
        Q[i, j] += 2 * PENALTY_A       # from A*2*sum_{i<j} x_i x_j

# Store for traceability
qubo_payload = {
    "scenario": "B",
    "n": int(n),
    "nct_ids": candidates["nct_id"].astype(str).tolist(),
    "params": {
        "K": int(K),
        "lambda_cost": float(LAMBDA_COST),
        "lambda_safety": float(LAMBDA_SAFETY),
        "penalty_A": float(PENALTY_A),
    },
    "components": {
        "_benefit_raw": benefit.tolist(),
        "_cost_raw": cost.tolist(),
        "_safety_raw": safety.tolist(),
        "weight_w": w.tolist(),
    },
    "Q": Q.tolist(),
}

with open(PATH_QUBO_JSON, "w") as f:
    json.dump(qubo_payload, f, indent=2)

print("Wrote QUBO JSON:", PATH_QUBO_JSON)
print("Q shape:", Q.shape)
print("Weight range w:", float(np.min(w)), "to", float(np.max(w)))


Wrote QUBO JSON: data/qubo_scenarios/scenario_B_qubo.json
Q shape: (60, 60)
Weight range w: 1.0 to 1.0


### What Cell 3 Just Did

- Defined the Scenario B portfolio selection objective (minimize negative benefit + weighted cost + weighted safety).
- Added a quadratic penalty enforcing an approximate “select K trials” constraint.
- Built the full QUBO matrix `Q` and wrote a traceable JSON including:
  - `nct_ids` ordering,
  - objective components,
  - and the final Q matrix used for classical and quantum solvers.


In [4]:
# ============================================================
# Cell 4 — Classical baseline solver (greedy local-search style)
# ============================================================

def qubo_energy(Q, x):
    x = np.asarray(x, dtype=float).reshape(-1, 1)
    return float((x.T @ Q @ x)[0, 0])

def greedy_fixK(Q, K):
    """
    Construct exactly-K solution by selecting K items with best marginal diagonal scores,
    then improving with single swaps.
    """
    n = Q.shape[0]
    # start from smallest diagonal entries (good under minimization)
    idx = np.argsort(np.diag(Q))[:K]
    x = np.zeros(n, dtype=int)
    x[idx] = 1

    bestE = qubo_energy(Q, x)
    improved = True

    while improved:
        improved = False
        ones = np.where(x == 1)[0]
        zeros = np.where(x == 0)[0]
        for i in ones:
            for j in zeros:
                x2 = x.copy()
                x2[i] = 0
                x2[j] = 1
                E2 = qubo_energy(Q, x2)
                if E2 < bestE:
                    x = x2
                    bestE = E2
                    improved = True
                    break
            if improved:
                break

    return x, bestE

x_best, E_best = greedy_fixK(Q, K=K)
bitstring = "".join(str(int(b)) for b in x_best.tolist())

selected_idx = np.where(x_best == 1)[0].tolist()
selected = candidates.iloc[selected_idx].copy()
selected.insert(0, "index", selected_idx)

# objective decomposition (original linear part, without penalty) for reporting
objective_linear = float(np.sum(w * x_best))
selected_n = int(np.sum(x_best))

summary = pd.DataFrame([{
    "scenario": "B",
    "method": "classical_greedy_fixK",
    "n_candidates": int(n),
    "K": int(K),
    "lambda_cost": float(LAMBDA_COST),
    "lambda_safety": float(LAMBDA_SAFETY),
    "penalty_A": float(PENALTY_A),
    "selected_n": selected_n,
    "qubo_energy": float(E_best),
    "linear_objective_only": float(objective_linear),
}])

summary.to_csv(PATH_CLASSICAL_SUMMARY, index=False)
selected.to_csv(PATH_CLASSICAL_SELECTED, index=False)
PATH_CLASSICAL_BITSTRING.write_text(bitstring)

print("Wrote classical baseline artifacts:")
print("  -", PATH_CLASSICAL_SUMMARY)
print("  -", PATH_CLASSICAL_SELECTED)
print("  -", PATH_CLASSICAL_BITSTRING)

display(summary)
display(selected.head(12))

Wrote classical baseline artifacts:
  - data/results/scenario_B_classical_summary.csv
  - data/results/scenario_B_classical_selected_trials.csv
  - data/results/scenario_B_classical_best_bitstring.txt


,scenario,method,n_candidates,K,lambda_cost,lambda_safety,penalty_A,selected_n,qubo_energy,linear_objective_only
0,B,classical_greedy_fixK,60,10,1.0,1.0,10.0,10,-990.0,10.0


,index,nct_id,brief_title,lead_sponsor,overall_status,phase,_benefit_raw,_cost_raw,_safety_raw,_pool_score
0,0,NCT06760637,Study of PF-07220060 With Letrozole in Adults ...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
32,32,NCT06742723,A Phase III Renal Outcomes and Cardiovascular ...,AstraZeneca,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
33,33,NCT05595642,A Study to Evaluate Astegolimab in Participant...,Hoffmann-La Roche,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
34,34,NCT05597540,Efficacy of 7 Days Versus 14 Days of Antibioti...,Assistance Publique - Hôpitaux de Paris,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
35,35,NCT05600491,A Phase III Study of Postoperative Early Temoz...,Sun Yat-sen University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
36,36,NCT06739122,A Study of Dulaglutide (LY2189265) 3.0 mg and ...,Eli Lilly and Company,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
37,37,NCT05609630,Study of Oral Upadacitinib and Subcutaneous/In...,AbbVie,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
38,38,NCT05609968,Study of Pembrolizumab (MK-3475) Monotherapy V...,Merck Sharp & Dohme LLC,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
39,39,NCT05611801,A Clinical Trial of Study Medicine (Marstacima...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
40,40,NCT06734702,Neoadjuvant Immunotherapy Plus Chemotherapy Fo...,Sun Yat-sen University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893


### What Cell 4 Just Did

- Solved the Scenario B QUBO classically using a deterministic greedy “exact-K then swap-improve” approach.
- Exported baseline artifacts (summary, selected trials, best bitstring) that downstream QAOA notebooks can compare against.


## Summary

Scenario B now has a complete optimization instance:

- `scenario_B_candidates.csv` → QUBO (`scenario_B_qubo.json`)
- Classical baseline → selected portfolio + objective score

Next:
- Run Scenario B robustness sweeps:
  - **05a-style greedy sweeps** (noise/λ/K sweeps)
  - **05b-style QAOA sweeps** (Aer + Aer-free fallback)
- Then produce a comparison notebook and a Braket SV1 run for Scenario B.
